In [2]:
import pandas as pd

In [3]:
estudiantes = pd.read_csv('data/dim_estudiante.tsv', sep="\t", encoding="latin1")
examenes = pd.read_csv('data/fact_examen_estudiante.tsv', sep="\t", encoding="latin1")
inscripciones = pd.read_csv('data/fact_inscripcion.tsv', sep="\t", encoding="latin1")
estudiantes.head()
inscripciones.head()

,inscripcion_skey,estudiante_skey,tiempo_skey,dictado_skey,estado,abandono
0,1,117876,20230213,1,Activa,0
1,2,117876,20230203,2,Baja,1
2,3,117876,20240209,4,Activa,0
3,4,117876,20230323,5,Activa,0
4,5,117876,20240321,8,Activa,0


In [13]:
df_inscripciones_por_alumno = (
    inscripciones.groupby("estudiante_skey")
    .agg(total_inscripciones=("inscripcion_skey", "count"))
    .reset_index()
)
df_inscripciones_por_alumno.head()

,estudiante_skey,total_inscripciones
0,1,10
1,2,6
2,3,6
3,4,7
4,5,10


In [14]:
df_fusion = pd.merge(
    estudiantes, df_inscripciones_por_alumno, on="estudiante_skey", how="left"
)
df_fusion.head()

,estudiante_skey,id_estudiante,dni,nombre,apellido,genero,fecha_nacimiento,nacionalidad,anio_ingreso,edad_ingreso,...,anio_abandono,nombre_programa,tipo_programa,duracion_programa,anio_plan_programa,facultad_programa,valid_from,valid_to,es_actual,total_inscripciones
0,1,45414,20000016,Gonzalo,Muñoz,M,2001-05-20,Venezolana,2023,22,...,NaN,Ingeniería Industrial,Grado,5,NaN,Facultad Regional Córdoba,2026-05-26,NaN,1,10.0
1,2,106375,20000125,Soledad,Suárez,F,2001-12-24,Argentina,2023,22,...,NaN,Ingeniería En Sistemas De Información,Grado,5,NaN,Facultad Regional Mendoza,2026-05-26,NaN,1,6.0
2,3,125211,20000181,Matías,García,M,2005-08-05,Argentina,2021,16,...,NaN,Ingeniería Industrial,Grado,5,NaN,Facultad Regional Córdoba,2026-05-26,NaN,1,6.0
3,4,28539,20000646,Maximiliano,Álvarez,M,2003-03-20,Argentina,2023,20,...,NaN,Ingeniería Mecánica,Grado,5,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,7.0
4,5,78953,20000756,Claudio,Sánchez,M,2007-04-02,Argentina,2020,13,...,NaN,Ingeniería En Sistemas De Información,Grado,5,NaN,Facultad Regional Rosario,2026-05-26,NaN,1,10.0


In [16]:
columna_hecho = (
    "total_inscripciones"  # <-- Cambialo por alguna columna propia de fact_inscripcion
)

df_fusion["es_ingresante_sin_cursada"] = (
    df_fusion[columna_hecho].isna().astype(int)
)
df_fusion.head()

,estudiante_skey,id_estudiante,dni,nombre,apellido,genero,fecha_nacimiento,nacionalidad,anio_ingreso,edad_ingreso,...,nombre_programa,tipo_programa,duracion_programa,anio_plan_programa,facultad_programa,valid_from,valid_to,es_actual,total_inscripciones,es_ingresante_sin_cursada
0,1,45414,20000016,Gonzalo,Muñoz,M,2001-05-20,Venezolana,2023,22,...,Ingeniería Industrial,Grado,5,NaN,Facultad Regional Córdoba,2026-05-26,NaN,1,10.0,0
1,2,106375,20000125,Soledad,Suárez,F,2001-12-24,Argentina,2023,22,...,Ingeniería En Sistemas De Información,Grado,5,NaN,Facultad Regional Mendoza,2026-05-26,NaN,1,6.0,0
2,3,125211,20000181,Matías,García,M,2005-08-05,Argentina,2021,16,...,Ingeniería Industrial,Grado,5,NaN,Facultad Regional Córdoba,2026-05-26,NaN,1,6.0,0
3,4,28539,20000646,Maximiliano,Álvarez,M,2003-03-20,Argentina,2023,20,...,Ingeniería Mecánica,Grado,5,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,7.0,0
4,5,78953,20000756,Claudio,Sánchez,M,2007-04-02,Argentina,2020,13,...,Ingeniería En Sistemas De Información,Grado,5,NaN,Facultad Regional Rosario,2026-05-26,NaN,1,10.0,0


In [17]:
# 4. Separamos los datasets como acordamos para la estrategia del proyecto
# Grupo A: Los 3.473 alumnos que se inscribieron a la carrera pero nunca registraron cursada inicial
df_riesgo_temprano = df_fusion[df_fusion["es_ingresante_sin_cursada"] == 1]

# Grupo B: Los alumnos que sí registraron cursada (con los que vas a entrenar el modelo de ML)
df_modelo = df_fusion[df_fusion["es_ingresante_sin_cursada"] == 0]

df_riesgo_temprano.head()

,estudiante_skey,id_estudiante,dni,nombre,apellido,genero,fecha_nacimiento,nacionalidad,anio_ingreso,edad_ingreso,...,nombre_programa,tipo_programa,duracion_programa,anio_plan_programa,facultad_programa,valid_from,valid_to,es_actual,total_inscripciones,es_ingresante_sin_cursada
24,25,114512,20006443,Rodrigo,Benítez,M,2004-12-16,Boliviana,2020,16,...,Ingeniería Química,Grado,5,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,NaN,1
130,131,14758,20030151,Sebastián,Ramos,M,1998-07-15,Argentina,2024,26,...,Tecnicatura Universitaria En Programación,Pregrado,3,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,NaN,1
148,149,89672,20033117,Iván,Romero,M,2007-08-22,Argentina,2020,13,...,Ingeniería Mecánica,Grado,5,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,NaN,1
203,204,76879,20043295,Valeria,Cabrera,F,2003-01-21,Argentina,2024,21,...,Tecnicatura Universitaria En Programación,Pregrado,3,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,NaN,1
235,236,96370,20047833,Julián,Souza,M,2005-09-11,Venezolana,2020,15,...,Ingeniería Aeronáutica,Grado,5,NaN,Facultad Regional Buenos Aires,2026-05-26,NaN,1,NaN,1


In [19]:
# 5. Guardamos el reporte del Grupo A para la acción institucional que planteaste en el Paso 1
df_riesgo_temprano.to_csv("data/alumnos_riesgo_temprano.csv", index=False)

# Verificamos los tamaños en el notebook para quedarnos tranquilos
print(f"Total de estudiantes en el DW: {len(estudiantes)}")
print(
    f"Alumnos en riesgo temprano: {len(df_riesgo_temprano)}"
)
print(f"Alumnos listos para el modelo analítico: {len(df_modelo)}")

Total de estudiantes en el DW: 127433
Alumnos en riesgo temprano: 3044
Alumnos listos para el modelo analítico: 124389


# Bloque 2

In [ ]:
# cambiar codigo_algoritmo
df_examen['es_algoritmos_reprobada'] = ((df_examen['id_materia'] == codigo_algoritmos) & (df_examen['es_reprobada'] == 1)).astype(int)

df_academicos = df_examen.groupby('id_estudiante').agg(
    promedio_notas=('nota', 'mean'),
    materias_reprobadas=('es_reprobada', 'sum'),
    reprobo_algoritmos=('es_algoritmos_reprobada', 'max')
).reset_index()

# 2. Agregamos el nivel máximo alcanzado desde las inscripciones/dictados (Fácil)
df_nivel = df_inscripcion.groupby('id_estudiante').agg(
    nivel_curso_maximo=('nivel_curso', 'max')
).reset_index()

# 3. Pegamos todo a la dimensión estudiante (df_modelo que filtramos en el paso 1)
df_master = pd.merge(df_modelo, df_academicos, on='id_estudiante', how='left')
df_master = pd.merge(df_master, df_nivel, on='id_estudiante', how='left')

# Rellenamos nulos por si las dudas (Garbage In, Garbage Out) [cite: 716]
df_master['promedio_notas'] = df_master['promedio_notas'].fillna(0)
df_master['materias_reprobadas'] = df_master['materias_reprobadas'].fillna(0)
df_master['reprobo_algoritmos'] = df_master['reprobo_algoritmos'].fillna(0)
df_master['nivel_curso_maximo'] = df_master['nivel_curso_maximo'].fillna(1) # Si no tiene nivel, asumimos ingresante